# Nemotron 3 Ultra in an Advisor/Handoff Workflow
Author: [Zain Hasan](https://x.com/ZainHasan6)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/togethercomputer/together-cookbook/blob/main/Agents/Cross_Provider_Advisor_Workflow.ipynb)


## Introduction

In this notebook we'll build an executor + advisor agent workflow that spans two providers. A strong, cost-effective executor on Together AI does the bulk of the work, and a higher-intelligence advisor on Anthropic gets pulled in only when the executor gets stuck.

There are two roles to fill:

| Role | Model | Provider | Job |
| --- | --- | --- | --- |
| **Executor** | `nvidia/nemotron-3-ultra-550b-a55b` | Together AI | Does the work, turn after turn |
| **Advisor** | `claude-opus-4-8` | Anthropic | Consulted only when the executor gets stuck |

The idea comes from the Minions work: the paper [*Minions: Cost-efficient Collaboration Between On-device and Cloud Language Models*](https://arxiv.org/abs/2502.15964) (Narayan et al., Stanford Hazy Research) and the companion [Ollama blog post](https://ollama.com/blog/minions). The core insight is that you can pair a cheap model that does most of the work with a frontier model you call sparingly, and cut the cost of the expensive model while keeping most of its quality:

- **Minion**: a cloud frontier model chats freely with a single local model that holds the data. This cuts remote cost by roughly 30.4x while keeping about 87% of frontier quality.
- **MinionS**: the cloud model breaks the task into small subtasks that run locally in parallel. This cuts remote cost by roughly 5.7x while keeping about 97.9% of frontier quality.

Here we take that same cost-efficient collaboration and run it across two hosted providers instead of across on-device and cloud. The executor runs on Together AI and generates the bulk of the tokens, and the frontier advisor runs on Anthropic and only gets consulted when the executor decides it's stuck. We wire the two together with ordinary function calling, so the executor can be any open model on Together while the advisor stays a frontier Claude model.

On long-horizon agentic work most turns are mechanical, but a good plan or a well-timed course correction is what really moves the needle. This setup gets you close to advisor-quality decisions while most of the generation happens at the cheaper executor's rates.

## Cross-Provider Advisor Agent Workflow

In this workflow a capable executor model handles a task on its own and only reaches for a stronger advisor model when it decides it's stuck. The executor owns the timing decision (when to ask for help), while our code owns the context (forwarding the full transcript) and the budget (capping how many times the advisor can be consulted).

Here's the overall flow:

<img src="../images/advisor.png" alt="Advisor Workflow Diagram" style="width: 65%; height: auto; display: block;">

1. We give the executor a single `advisor` tool. It takes no parameters; the executor is just signaling "I need help now."
2. With `tool_choice="auto"` the executor decides whether to call it. That's the whole point: the advisor is a last resort, not a crutch.
3. When the executor calls `advisor`, our code forwards the whole conversation so far to Claude Opus, gets back a short strategic note, and feeds it back as the tool result.
4. The executor picks up where it left off, now with the advice in hand.

Like the Minions protocol, which leans on orchestration code to mediate the local and cloud exchange, our code is the bridge that stitches the two providers together. It forwards the executor's full transcript to the advisor and feeds the advice back as a tool result. The advisor tool's `input` is intentionally empty: the executor only signals when to ask, and we supply the what.

Now let's look at the implementation.


## Setup and Utils

You need two API keys: one for the [Together-hosted](https://api.together.ai/) executor and one for the [Anthropic](https://console.anthropic.com/) advisor.

In [1]:
# Install libraries
!uv pip install -U together anthropic

Using Python 3.12.8 environment at: /opt/anaconda3/envs/cookbook
⠹ markdown-it-py==4.2.0                                                         Resolved 34 packages in 331ms
Audited 34 packages in 0.51ms


In [2]:
# Import libraries
import os
from together import Together
from anthropic import Anthropic

together  = Together(api_key=os.getenv("TOGETHER_API_KEY"), timeout=1800.0)
anthropic = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY")) # Anthropic hosts the advisor model

In [3]:
# Nemotron Ultra is a reasoning model: a single call can spend many minutes and
# tens of thousands of tokens thinking, so give the HTTP client a long timeout.
EXECUTOR_MODEL = "nvidia/nemotron-3-ultra-550b-a55b"   # Together AI
ADVISOR_MODEL  = "claude-opus-4-8"                      # Anthropic
MAX_ADVISOR_CALLS = 3                                   # conversation-level cap (enforced client-side)

## Advisor Agent Implementation

Here's the plan for what we'll build: define the advisor tool, build a bridge that forwards the executor's transcript to the advisor, and wrap both in an executor loop that decides when to consult.

### Step 1: Define the advisor tool

The tool intentionally has no parameters (`"properties": {}`). The executor doesn't write a question; it just decides when to reach for help, and we supply the context ourselves.

In [4]:
ADVISOR_TOOL = {
    "type": "function",
    "function": {
        "name": "advisor",
        "description": (
            "Consult a stronger reviewer model for strategic guidance. Takes NO "
            "parameters -- your full conversation history is forwarded automatically. "
            "PRECONDITION: only call this AFTER you have made a real attempt and have "
            "concrete work to show -- a draft solution, a specific approach you are "
            "unsure about, or an error you cannot resolve. Calling it before attempting "
            "the task wastes it: there is nothing for the reviewer to react to. Do NOT "
            "call it for routine steps you can handle yourself, to ask permission, or to "
            "confirm something you already believe is correct."
        ),
        "parameters": {"type": "object", "properties": {}},
    },
}

SYSTEM_PROMPT = (
    "You are a capable engineer. Default to solving the task yourself, end to end. "
    "Always make a genuine attempt FIRST -- think it through and produce a concrete "
    "draft before considering any tool.\n\n"
    "You have one escape hatch: an `advisor` tool backed by a stronger model. It is a "
    "last resort, not a first step. Only call it when you have already attempted the "
    "task and are genuinely stuck -- errors keep recurring, your approach is not "
    "converging, or you are about to commit to a hard-to-reverse decision you cannot "
    "resolve on your own. Do NOT call it before attempting the task, to ask for "
    "permission, or to double-check work you are already fairly confident in. "
    "If you can answer, just answer."
)

### Step 2: The advisor bridge (Together to Anthropic)

When the executor calls `advisor`, this function rebuilds the transcript and asks Claude Opus for a focused note. Three things keep the bridge reliable:

- **Separate the task from the work, with clear role labels** (`TASK`, `EXECUTOR`, `YOUR EARLIER ADVICE`), and strip the raw `advisor()` tool-call plumbing out of the history. Without this the advisor reads the bare `advisor()` calls as a "delegation loop," decides it's the one being supervised, and replies with meta-commentary like "just implement!" instead of real guidance. That's exactly the failure we hit on the first run.
- **Push for task-focused guidance.** The prompt tells the advisor to give the concrete approach to the task and not to critique the executor. One substantive consult is then usually enough, instead of the executor bouncing off useless advice and calling again.
- **Ask for brevity** (under 120 words). The advisor's output is the main cost driver, and a focused starting point beats a comprehensive plan.

In [ ]:
def consult_advisor(messages):
    """Forward the executor's transcript to the Opus advisor and return a short note."""
    # Separate the task (first user turn) from the executor's own work so the advisor
    # always knows what problem to advise on.
    task = next((m["content"] for m in messages if m["role"] == "user"), "")

    # Build a clean, role-labelled transcript of the executor's work. We deliberately
    # DROP the raw `advisor()` tool-call plumbing and clearly mark our own prior advice,
    # otherwise the advisor misreads the history as a "delegation loop" and ends up
    # scolding the executor ("just implement!") instead of giving task guidance.
    transcript = []
    for m in messages:
        role = m["role"]
        if role in ("system", "user"):
            continue
        if role == "assistant":
            content = (m.get("content") or "").strip()
            if content:
                transcript.append(f"EXECUTOR: {content}")
        elif role == "tool":
            note = (m.get("content") or "").strip()
            transcript.append(f"YOUR EARLIER ADVICE: {note}")
    work = "\n\n".join(transcript) or (
        "(The executor asked for help immediately, before producing any work.)"
    )

    advice = anthropic.messages.create(
        model=ADVISOR_MODEL,
        max_tokens=32000,   # Anthropic requires this field; set high so advice is never truncated
        messages=[{
            "role": "user",
            "content": (
                "You are a senior advisor helping a faster 'executor' model that asked "
                "for guidance. Give concrete, technical direction on how to SOLVE THE "
                "TASK: the key insight, the right approach, or the specific course "
                "correction. Do NOT critique the executor's behaviour or tell it to "
                "'just implement' -- assume it will act on substantive direction. Keep "
                "it under 500 words: a focused starting point, not a comprehensive plan.\n\n"
                f"=== TASK ===\n{task}\n\n"
                f"=== EXECUTOR'S WORK SO FAR ===\n{work}"
            ),
        }],
    )
    return "".join(b.text for b in advice.content if b.type == "text")

### Step 3: The executor loop

This is a standard function-calling loop. The one twist is that we count advisor calls, and once we hit the cap we drop the tool from `tools` and set `tool_choice` to `"none"`. Together doesn't have a built-in conversation-level cap, so we enforce one ourselves.

One thing to watch with reasoning models: Nemotron Ultra spends most of its budget thinking. If you give it a small `max_tokens` it can burn through the budget before emitting any `content` (you'll see `finish_reason="length"` and an empty answer). We set a large `max_tokens` so there's room for both the reasoning and the final text.

In [ ]:
def run(task: str, verbose: bool = True, max_turns: int = 12):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": task},
    ]
    advisor_calls = 0
    msg = None

    for turn in range(1, max_turns + 1):
        # Drop the advisor tool once the cap is hit so the executor stops trying.
        capped = advisor_calls >= MAX_ADVISOR_CALLS
        if verbose:
            print(f"[turn {turn}] executor working{' (advisor budget spent)' if capped else ''}...")
        kwargs = dict(
            model=EXECUTOR_MODEL,
            messages=messages,
            max_tokens=130000,   # reasoning model: leave ample room for thinking + content
        )
        if not capped:  # only advertise the tool (and tool_choice) while budget remains
            kwargs["tools"] = [ADVISOR_TOOL]
            kwargs["tool_choice"] = "auto"

        resp = together.chat.completions.create(**kwargs)
        msg = resp.choices[0].message
        messages.append(msg.model_dump())

        if not msg.tool_calls:
            if verbose:
                print(f"[turn {turn}] executor produced the final answer.\n")
            return msg.content   # executor is done

        if verbose:
            print(f"[turn {turn}] executor paused to call: "
                  f"{', '.join(c.function.name + '()' for c in msg.tool_calls)}")

        for call in msg.tool_calls:
            if call.function.name == "advisor" and not capped:
                advisor_calls += 1
                if verbose:
                    print(f"  >>> executor consulted the advisor (call #{advisor_calls})")
                result = consult_advisor(messages)
                if verbose:
                    print(f"  <<< opus advice:\n{result}\n")
            elif call.function.name == "advisor":
                # Budget spent: don't silently bypass the cap, tell it to finish up.
                result = "Advisor budget exhausted. Finalize your answer yourself now."
            else:
                result = f"Unknown tool: {call.function.name}"

            messages.append({
                "role": "tool",
                "tool_call_id": call.id,
                "name": call.function.name,
                "content": result,
            })

    return (msg.content if msg else None) or "(stopped: hit max_turns without a final answer)"

### Step 4: Run it

Let's give the executor a task it can handle on its own. Depending on the model it might solve the task directly or reach for the advisor first. Either way, the bridge makes sure any consult comes back with useful, task-focused guidance. The next section digs into what actually happens on each run.

In [60]:
task = """Implement a thread-safe sliding-window rate limiter in Python: allow at most N 
requests per rolling T-second window, bursts must never exceed N, and it must be correct 
under heavy multithreading. Think about the tricky edge cases."""

answer = run(task)
print(answer[:1500])

[turn 1] executor working...
[turn 1] executor paused to call: advisor()
  >>> executor consulted the advisor (call #1)
  <<< opus advice:
Use a `collections.deque` of timestamps guarded by a single `threading.Lock`. Core logic in `allow()`:

1. Acquire lock.
2. `now = time.monotonic()` — use monotonic, never `time.time()` (wall-clock jumps break correctness).
3. Evict from the left while `timestamps[0] <= now - T`.
4. If `len(timestamps) < N`: append `now`, return True; else return False.

Key correctness points:
- The eviction + check + append must all happen inside the same lock hold; don't release between them or two threads can both see room and overflow N.
- Use `<=` for eviction so a request exactly T seconds old is expired (true *sliding* window, not fixed buckets — that's what bounds bursts to N over any T-window).
- Keep the critical section tiny; no sleeping/blocking under the lock.

If you later want blocking acquisition, compute wait time as `timestamps[0] + T - now`, slee

### A harder task

To see what the executor decides, we wrap the Together call with a small tracer. For each turn it prints whether the advisor was offered, whether it was called, and how the token budget split between reasoning and content.

In [59]:
_orig_create = together.chat.completions.create
_turn = {"n": 0}

def _traced(**kw):
    _turn["n"] += 1
    offered = any(t["function"]["name"] == "advisor" for t in (kw.get("tools") or []))
    r = _orig_create(**kw)
    m = r.choices[0].message
    print(
        f"turn {_turn['n']}: finish={r.choices[0].finish_reason} "
        f"called_advisor={bool(m.tool_calls)} "
        f"content_tokens~={len(m.content or '')//4} "
        f"reasoning_tokens~={len(getattr(m, 'reasoning', None) or '')//4} "
        f"advisor_offered={offered}"
    )
    return r

together.chat.completions.create = _traced
try:
    hard_task = (
        "I have up to 200000 axis-aligned, possibly-overlapping rectangles. I need the "
        "EXACT area of their union in O(N log N). Decide the algorithm and justify the "
        "complexity BEFORE coding. If you are not fully certain it is genuinely "
        "O(N log N) and handles all degenerate overlaps, consult the advisor first."
    )
    _ = run(hard_task, verbose=True)
finally:
    together.chat.completions.create = _orig_create

[turn 1] executor working...
turn 1: finish=tool_calls called_advisor=True content_tokens~=644 reasoning_tokens~=845 advisor_offered=True
[turn 1] executor paused to call: advisor()
  >>> executor consulted the advisor (call #1)
  <<< opus advice:
Your plan is correct and genuinely O(N log N). One critical subtlety to nail before coding:

**The segment tree update must NOT do lazy propagation in the usual sense.** This is a "coverage" segment tree where `cover` counts are stored at the exact nodes a range-update touches (the canonical O(log N) decomposition nodes), and `length` is recomputed bottom-up. Don't push `cover` down to children — a node with `cover>0` reports full length regardless of children's state. After updating `cover` at touched nodes, recompute `length` on the way back up using the rule:
- `cover>0` → full length
- leaf → 0
- else → sum of children

This handles partial overlaps correctly. Proceed.

[turn 2] executor working...
turn 2: finish=tool_calls called_advisor

## Cost and tuning notes

- **Conversation-level cap.** Together doesn't impose a built-in limit on tool calls, so we count `advisor` calls and drop the tool once we hit `MAX_ADVISOR_CALLS`. If your provider validates message history, don't just stop passing the tool while leaving prior tool results in the history; double-check that the history stays consistent.
- **Advisor brevity is the main lever.** The advisor's output is billed at Opus rates. The "under 120 words" request in the prompt keeps it focused, so tighten or loosen it per task.
- **Two separate token budgets.** The executor's `max_tokens` (130k here) bounds Nemotron only. Anthropic requires a `max_tokens` on its call, so we set it high (32k) to avoid truncating the advice. The "under 120 words" instruction in the prompt is what actually keeps it short.
- **Tune when it fires through the tool description**, not in code. "Last resort" gives you a rare safety net. "Before committing, and again before done" gives you roughly two consults per task, which lines up with typical agentic coding guidance.

## Takeaways

- You can take the Minions cost-efficient collaboration pattern and run it across two hosted providers with plain function calling: a Together-hosted open model as the executor and a frontier Claude model as the advisor, instead of the original on-device and cloud split.
- The executor owns the timing decision, while your code owns the context (forwarding the full transcript) and the budget (capping the calls).
- With a strong reasoning executor like Nemotron Ultra, the advisor ends up being a rarely-used safety net, which is exactly what "only when it can't figure it out itself" should look like.
